# GOT-OCR 2.0 — DIMER optical character recognition tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/got-ocr2-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/got-ocr2-pipeline/blob/main/tutorials/got_ocr2_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-stepfun--ai%2FGOT--OCR--2.0--hf-ffcc4d?style=flat)](https://huggingface.co/stepfun-ai/GOT-OCR-2.0-hf) [![Upstream](https://img.shields.io/badge/Upstream-Ucas--HaoranWei%2FGOT--OCR2.0-181717?style=flat&logo=github&logoColor=white)](https://github.com/Ucas-HaoranWei/GOT-OCR2.0) [![arXiv](https://img.shields.io/badge/arXiv-2409.01704-b31b1b.svg)](https://arxiv.org/abs/2409.01704)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** optical character recognition — one image → its text, plain or formatted (Markdown/LaTeX) — using the pinned `stepfun-ai/GOT-OCR-2.0-hf` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/got_ocr2_pipeline/pipeline.py` at revision `c3ac53275b89`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `d3017ef2c2c1395888c8d635c5e0508bcb0ac78d` (~1140 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the GOT-OCR 2.0 model — a ViT-style vision encoder over a 1024×1024 image producing 576 image tokens and a 24-layer Qwen2-architecture decoder (about 580M parameters) — reads the image wrapped in the snapshot's chat prompt for the chosen mode and generates the text token by token until the `<|im_end|>` stop string or the budget. In `plain` mode the output is the page text with line breaks; in `format` mode it is formatted text (Markdown/LaTeX constructs such as `\title{}`, tables, formulas). Decoding is greedy (`do_sample=False`) under a caller-owned `max_new_tokens` budget. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, one of two modes, the token budget), a fixed output contract, and the `character_error_rate`, `word_error_rate`, `validate_inputs` and `evaluation_report` helpers. The default sample is a notice page rendered in code from known text, so its transcript is the reference; the resulting error rates are demonstration (plumbing) evidence for one clean rendered page, not an OCR benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, render a synthetic printed page with a known transcript (or upload your own image) and validate it into an input manifest, choose a recognition mode and a token budget, run the supported task, read the output correctly (generated text, no score, a `truncated` flag), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `character_error_rate` and `word_error_rate` only when a reference transcript exists and `not-measurable` otherwise, and export the text and provenance.

**This notebook does not demonstrate:** PDF or multi-page documents (one image per call; the upstream multi-page mode is not exposed), region-guided OCR by box or colour (upstream interactive modes, not exposed), rendering the formatted output (LaTeX, Markdown tables or charts need external tools), layout or reading-order output (only the text stream is returned), batch throughput, sampling or beam search, evaluation on an OCR benchmark (which needs labelled images; only the rendered page is scored here), and any training. The model generates text for any image, including one with no text, and gives no signal when it invents.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (bfloat16, the checkpoint's stored dtype). CPU is adequate but slow: the repository's model card records 6.7 s to load and 11.4 s for the 1000×520 rendered page (154 generated tokens) in the Windows venv (Intel Core Ultra 9 275HX); cost scales with the tokens generated. The pinned `torch==2.14.0` install and the 1.12 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what a vision–language model's generated tokens are; what character and word error rate measure; that fluent output is not correct output.
- **Data:** the default sample is a deterministic 1000×520 notice page rendered in code with Pillow's bundled font — a heading and seven lines of English text with dates, numbers and punctuation — so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar) containing text — a document page, a photograph of a sign, a screenshot — any colour mode, sides between 16 and 4096 px. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `stepfun-ai/GOT-OCR-2.0-hf` snapshot (~1140 MB in total) at revision `d3017ef2c2c1…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'got-ocr2-pipeline',
    'repository_revision': 'c3ac53275b897136a209e83129145b0e31a2f7fc',
    'embedded_module': 'src/got_ocr2_pipeline/pipeline.py',
    'embedded_modules': ['src/got_ocr2_pipeline/pipeline.py'],
    'module_sha256': 'f4b0e3330480550595956200c63c084fe49433368628bdc0ca80b9f32ed6e971',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/got_ocr2_pipeline/` @ `c3ac53275b89`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/got_ocr2_pipeline/pipeline.py`

In [ ]:
"""Optical character recognition with the pinned ``stepfun-ai/GOT-OCR-2.0-hf`` checkpoint (GOT-OCR 2.0).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the GOT-OCR2 architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "stepfun-ai/GOT-OCR-2.0-hf"
MODEL_REVISION = "d3017ef2c2c1395888c8d635c5e0508bcb0ac78d"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "got-ocr-2.0-hf"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The two recognition modes the pinned README documents: "plain" (the processor's default prompt,
# plain text) and "format" (processor(..., format=True): formatted text such as Markdown or LaTeX for
# tables and formulas). Region ("box"/"color") and multi-page modes are not exposed.
MODES = ("plain", "format")
DEFAULT_MODE = MODES[0]
# Generation ceilings. 4096 is the max_new_tokens the pinned README's examples pass; the default is a
# practical single-page budget.
MAX_NEW_TOKENS = 4096
DEFAULT_MAX_NEW_TOKENS = 1024
DECODING = "greedy"
STOP_STRING = "<|im_end|>"
# Input ceilings. The processor resizes every image to 1024x1024 (preprocessor_config.json, aspect
# ratio not preserved) into 576 image tokens, so image cost is bounded; the side ceiling only guards
# memory during decoding and resizing.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _tokens(text: str) -> list[str]:
    return text.lower().split()


def word_error_rate(reference: str, hypothesis: str) -> float:
    """Word error rate of ``hypothesis`` against ``reference`` after lower-casing and whitespace tokenisation.

    Levenshtein edits over words divided by reference words; punctuation is **not** stripped, so a
    stray comma counts. The metric a caller would use to score ``doctags_to_text`` against a known page.
    """
    ref, hyp = _tokens(reference), _tokens(hypothesis)
    if not ref:
        raise ValueError("reference must contain at least one word")
    previous = list(range(len(hyp) + 1))
    for row_index, ref_token in enumerate(ref, 1):
        current = [row_index]
        for column_index, hyp_token in enumerate(hyp, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[column_index] + 1,
                    previous[column_index - 1] + (ref_token != hyp_token),
                )
            )
        previous = current
    return previous[-1] / len(ref)


def character_error_rate(reference: str, hypothesis: str) -> float:
    """Character error rate of ``hypothesis`` against ``reference`` after whitespace normalisation.

    Levenshtein edits over characters divided by reference characters; case and punctuation are kept.
    The metric a caller would use to score OCR output against a known transcript.
    """
    ref, hyp = " ".join(reference.split()), " ".join(hypothesis.split())
    if not ref:
        raise ValueError("reference must contain at least one character")
    previous = list(range(len(hyp) + 1))
    for row_index, ref_char in enumerate(ref, 1):
        current = [row_index]
        for column_index, hyp_char in enumerate(hyp, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[column_index] + 1,
                    previous[column_index - 1] + (ref_char != hyp_char),
                )
            )
        previous = current
    return previous[-1] / len(ref)


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB): a page, a scene with text, a crop",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "modes": list(MODES),
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False) until {STOP_STRING} or the budget; deterministic per device",
    "preprocessing": (
        "image converted to RGB; the processor resizes to 1024x1024 (aspect ratio not preserved, CLIP "
        "mean/std) into 576 image tokens and wraps them in the snapshot's chat prompt for the chosen mode"
    ),
    "output": "the recognised text (plain, or formatted Markdown/LaTeX in format mode); no score",
}


def _check_inputs(image: Any, mode: Any, max_new_tokens: Any) -> tuple[Image.Image, str, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``recognize`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if not isinstance(mode, str):
        raise TypeError("mode must be a str")
    if mode not in MODES:
        raise ValueError(f"mode {mode!r} is not one of MODES {MODES}")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, mode, max_new_tokens


def validate_inputs(
    image: Image.Image,
    *,
    mode: str = DEFAULT_MODE,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``recognize`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked_mode, checked_tokens = _check_inputs(image, mode, max_new_tokens)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (recognize takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "recognition_mode": checked_mode,
        "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_text: str | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_text`` (the transcript the image really carries) the report carries
    ``character_error_rate`` and ``word_error_rate`` of the recognised text against it, verdict
    ``sample-sanity``; without it the verdict is ``not-measurable`` and the report says what labelled
    data would make the task measurable.
    """
    text = str(result["text"])
    base = {
        "task": "optical character recognition: image -> text",
        "score_semantics": (
            "generated text carries no score, no probability and no correctness signal; fluent output is "
            "not evidence that it matches the image. Greedy decoding makes the output reproducible on a "
            "fixed device and dtype, which is a reproducibility property, not a quality one"
        ),
        "recognition_mode": result.get("mode"),
        "sample_kind": sample_kind,
        "n_chars": len(text),
        "truncated": result.get("truncated"),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not reference_text:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference transcript was supplied for the evaluated image",
            "needs": (
                "images with ground-truth transcripts from the deployment domain (scans, scenes, fonts, "
                "languages) scored with character and word error rate; no such labelled set ships with this "
                "repository"
            ),
        }
    metrics = [
        {
            "id": "character_error_rate",
            "value": character_error_rate(reference_text, text),
            "normalisation": "whitespace collapsed; case and punctuation kept",
            "estimation": "one image, no dispersion estimate",
        },
        {
            "id": "word_error_rate",
            "value": word_error_rate(reference_text, text),
            "normalisation": "lower-cased, whitespace-tokenised; punctuation kept",
            "estimation": "one image, no dispersion estimate",
        },
    ]
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            "2 sanity measures on one tutorial image whose text you rendered yourself; plumbing evidence, "
            "not an OCR benchmark"
        ),
        "needs": (
            "a labelled image set from the deployment domain (scans, photographs, fonts, scripts, layouts) "
            "for any OCR accuracy claim"
        ),
    }


@dataclass
class GotOcr2Pipeline:
    """``_runner(image, mode, max_new_tokens)`` returns ``{"text": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> GotOcr2Pipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        dtype = torch.bfloat16 if resolved_device.startswith("cuda") else torch.float32
        processor = AutoProcessor.from_pretrained(location, **common)
        model = AutoModelForImageTextToText.from_pretrained(location, dtype=dtype, **common)
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, mode: str, max_new_tokens: int) -> dict[str, Any]:
            inputs = processor(image, return_tensors="pt", format=(mode == "format")).to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    do_sample=False,
                    tokenizer=processor.tokenizer,
                    stop_strings=STOP_STRING,
                    max_new_tokens=max_new_tokens,
                )
            new_ids = generated[0, inputs["input_ids"].shape[1] :]
            decoded = processor.decode(new_ids, skip_special_tokens=True)
            return {"text": decoded, "new_tokens": int(new_ids.shape[0])}

        return cls(runner, resolved_device, str(dtype).removeprefix("torch."), source)

    def recognize(
        self,
        image: Image.Image,
        *,
        mode: str = DEFAULT_MODE,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Recognise the text in one image; ``text`` is the decoded output with the stop string removed."""
        rgb, checked_mode, checked_tokens = _check_inputs(image, mode, max_new_tokens)
        raw = self._runner(rgb, checked_mode, checked_tokens)
        if not isinstance(raw, dict) or "text" not in raw:
            raise RuntimeError("runner must return a dict with 'text'")
        text = str(raw["text"]).replace(STOP_STRING, "").strip()
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "text": text,
            "mode": checked_mode,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `d3017ef2c2c1…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `GotOcr2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "got-ocr-2.0-hf",
  "modelId": "stepfun-ai/GOT-OCR-2.0-hf",
  "revision": "d3017ef2c2c1395888c8d635c5e0508bcb0ac78d",
  "files": [
    {
      "path": "README.md",
      "bytes": 12077,
      "sha256": "706d6fe217d2f047ca68f47bdcf44ace4a0832491da396972c2c917944e84c18"
    },
    {
      "path": "config.json",
      "bytes": 608,
      "sha256": "cbe8aacd6cd84a2d58eafcd0045c6ac40e02e3a448f24b8cee51cc81d8bdccf2"
    },
    {
      "path": "generation_config.json",
      "bytes": 74,
      "sha256": "31915c5a692f43c5765a20cfc5f9403bcd250f5721a0d931bb703169c08993b4"
    },
    {
      "path": "model.safetensors",
      "bytes": 1121114488,
      "sha256": "6175ac7868a4e75735f5d59f78c465081ad3427eb4f312d072a0f1d16b333ba4"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 439,
      "sha256": "ef9a0dc0935cac11f4230ca30d00a52bedfa52b6633e409e9fbd2ea56373aa7e"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 213,
      "sha256": "7c2368a3889fdfb37c24cabeb031b53f47934f357b54e56e8e389909a338ea47"
    },
    {
      "path": "tokenizer.json",
      "bytes": 18702549,
      "sha256": "36b382a3c48c9a143c30139dac6c8230ddfb0b46a3dc43082af6052abe99d9de"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 39228,
      "sha256": "8b0542937d32a67da8ea2d1288b870e325be383a962c65d201864299560a2b8e"
    }
  ],
  "totalBytes": 1139869676
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = GotOcr2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Render the synthetic printed page or optional BYOD

The default sample is **synthetic** and carries its own reference: a notice page — a heading in a larger face and seven lines of English text with dates, numbers, a postcode and punctuation — is rendered with Pillow's bundled font at 1000×520, the same page the repository's smoke run used. The lines drawn on it, joined with newlines, are the reference transcript for the error-rate sanity check later; they are not a labelled dataset, so nothing here is an OCR benchmark, and a rendered page is far cleaner than any scan or photograph. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one image — no transcript exists for it, so the evaluation report will be `not-measurable`.

The recognition mode and the token budget are **caller-owned request parameters**: `mode` is `plain` (the page text with line breaks) or `format` (formatted Markdown/LaTeX output, the upstream `format=True` path), and `max_new_tokens` bounds the generation (`DEFAULT_MAX_NEW_TOKENS = 1024`; `MAX_NEW_TOKENS = 4096` is the ceiling the pinned README's examples use). Nothing is validated in this cell — the next section hands the image and the request to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the page size and digest, the request, and the number of reference characters.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
mode = 'plain'  # @param ['plain', 'format']
max_new_tokens = 1024  # @param {type:"integer"}

LINES = [
    'NOTICE OF ANNUAL GENERAL MEETING',
    'The annual general meeting of Northwind Traders Ltd. will be held',
    'on Thursday 23 April 2026 at 10:00 in the Harbour Room, Portsmouth.',
    "Agenda: 1. Minutes of the previous meeting. 2. Directors' report.",
    '3. Approval of accounts for the year ended 31 December 2025.',
    '4. Re-election of directors. 5. Appointment of auditors.',
    'Shareholders unable to attend may appoint a proxy by 21 April 2026.',
    'Registered office: 14 Harbour Road, Portsmouth PO1 3AX. Reg. no. 04471120',
]


def printed_page(width=1000, height=520):
    """A notice page rendered with Pillow's bundled font; returns page + reference transcript."""
    page = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(page)
    head, body = ImageFont.load_default(size=30), ImageFont.load_default(size=22)
    d.text((60, 40), LINES[0], fill='black', font=head)
    y = 110
    for line in LINES[1:]:
        d.text((60, y), line, fill=(20, 20, 20), font=body)
        y += 48
    return page, '\n'.join(LINES)


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    reference_text = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic page: no randomness, so no seed is needed and the digest is stable per Pillow build.
    image, reference_text = printed_page()
    image_name = 'synthetic_notice_1000x520.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'recognition_mode': mode, 'max_new_tokens': max_new_tokens, 'reference_chars': None if reference_text is None else len(reference_text)})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `recognize` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, a mode that is one of `MODES`, and `max_new_tokens` in `[1, MAX_NEW_TOKENS]` — and returns an **input manifest** naming the schema (including the preprocessing the processor applies and the decoding rule), the input's observed mode and size, the request, and the verdict. The manifest is written to `outputs/got_ocr2_input_manifest.json`. To show what rejection looks like, the cell also validates a mode the pipeline does not expose and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB and resized to 1024×1024 by the processor (the aspect ratio is not preserved); nothing else is dropped or altered. The pipeline cannot tell whether the image contains text: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'MODES': list(MODES), 'DECODING': DECODING, 'STOP_STRING': STOP_STRING}})
input_manifest = validate_inputs(image, mode=mode, max_new_tokens=max_new_tokens, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(image, mode='multi-page')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'unsupported-mode-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/got_ocr2_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Recognise the text and read the output correctly

`recognize` returns a dict with `text` — the decoded output with the stop string removed — the `mode`, `image_size`, `new_tokens`, a `truncated` flag that is true when the budget was exhausted, the generation settings and the model identity. **No score exists**: generated text carries no probability and no correctness signal, and fluent output is not evidence that it matches the image. Greedy decoding is deterministic on a fixed device and dtype; CUDA bfloat16 can change a token and therefore the rest of the sequence, so GPU and CPU outputs need not match. As recorded in the model card, the repository's CPU smoke on this same page in `plain` mode generated 154 tokens in 11.4 s and reproduced the transcript **exactly** (character and word error rate 0.0); `format` mode wrapped the heading in `\title{}` and scored 0.02 / 0.025 against the plain transcript because of that markup; a blank 1024×1024 image produced `(0) = 1 +`. Those are observations on one rendered page, not calibration points.

In [ ]:
import time

t0 = time.time()
result = pipe.recognize(image, mode=mode, max_new_tokens=max_new_tokens)
elapsed = time.time() - t0
print({'seconds': round(elapsed, 1), 'new_tokens': result['new_tokens'], 'truncated': result['truncated'], 'device': pipe.device, 'dtype': pipe.dtype, 'n_chars': len(result['text'])})
print(result['text'][:1500] + ('…' if len(result['text']) > 1500 else ''))
if result['truncated']:
    print('The token budget was exhausted: the text is incomplete. Raise max_new_tokens (ceiling MAX_NEW_TOKENS) and rerun.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No OCR metric is reported by default: accuracy needs images with ground-truth transcripts from the deployment domain, and this repository ships none. The repository's metric helpers are `character_error_rate` (character-level Levenshtein distance over whitespace-normalised text, case and punctuation kept, divided by the reference length) and `word_error_rate` (word-level, lower-cased); when a reference transcript is supplied the report carries both, with the verdict `sample-sanity`. On the synthetic path that reference is text **you rendered yourself** on a clean white page, so a zero error rate proves only that the input contract, forward pass, decoding and stop handling round-trip; in `format` mode the added markup counts as errors against the plain transcript, which is expected. On BYOD no reference exists, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/got_ocr2_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, reference_text, sample_kind=sample_kind)
with open('outputs/got_ocr2_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k != 'metrics'}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:22} {metric['value']:.4f}  ({metric['normalisation']})")
if report['verdict'] == 'not-measurable':
    print('No reference transcript exists for this input, so nothing is scored; read the text against the image yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (text, mode, `new_tokens`, `truncated`, the budget), the evaluation report, the input manifest, the sample identity, digest and reference transcript, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device, dtype). The recognised text is also written verbatim to `outputs/got_ocr2.txt`, and a side-by-side PNG places the input image above a panel with the recognised text for visual inspection — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
panel_lines = result['text'].splitlines()[:24] or ['(no text)']
panel_height = 24 + 22 * len(panel_lines)
annotated = Image.new('RGB', (image.width, image.height + panel_height), 'white')
annotated.paste(image.convert('RGB'), (0, 0))
draw = ImageDraw.Draw(annotated)
draw.line([(0, image.height + 1), (image.width, image.height + 1)], fill=(120, 120, 120), width=2)
panel_font = ImageFont.load_default(size=16)
for index, line in enumerate(panel_lines):
    draw.text((16, image.height + 10 + 22 * index), line[:140], fill=(40, 90, 220), font=panel_font)
annotated.save('outputs/got_ocr2_annotated.png')
with open('outputs/got_ocr2.txt', 'w', encoding='utf-8') as handle:
    handle.write(result['text'])
payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'reference_text': reference_text},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': pipe.dtype,
    },
}
with open('outputs/got_ocr2_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The text is what the model generates after reading the image; nothing in the output scores it, and a fluent transcript can still carry wrong characters, skipped lines or invented content. On the synthetic page the error rates in the evaluation report compare the output with text you rendered yourself on a clean white page and the verdict is `sample-sanity`, which proves only that the input contract, forward pass, decoding and stop handling work (the repository's smoke run reproduced this page exactly in `plain` mode); they say nothing about scans, photographs, handwriting, low resolution, dense multi-column layouts, non-Latin scripts, formulas or tables, and a BYOD result is a single-image observation with the verdict `not-measurable`. **The model generates text for any image** and stops only at the stop string or the budget: a blank page yielded `(0) = 1 +` in the smoke run, so an image with no text produces an invented fragment rather than an empty result, and `truncated` is the only structural signal you get. Everything is resized to 1024×1024 without preserving the aspect ratio, so a wide receipt or a tall page is distorted. The pipeline provides no layout, no boxes, no multi-page or region modes, no rendering of formatted output and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** switch `mode` to `format` and see the heading come back as `\title{...}` (and the error rates rise for the markup alone); lower `max_new_tokens` to 40 and watch `truncated` turn true; downscale the page to 400 px wide before recognition and compare the error rates; enable `USE_BYOD` with a photograph of a sign or a scanned page, then type its transcript and pass it as `reference_text` to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/got-ocr2-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/got-ocr2-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/got-ocr2-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/stepfun-ai/GOT-OCR-2.0-hf
- Upstream code: https://github.com/Ucas-HaoranWei/GOT-OCR2.0
- General OCR Theory: Towards OCR-2.0 via a Unified End-to-end Model (Wei et al., 2024): https://arxiv.org/abs/2409.01704